# release16_failedfold_participation_target_overlay_v1

Notebook-first target overlay for the release16 failed folds.

- **Canonical baseline**: retained SOXL harness model frontier with `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02` as the buy/sell reference baseline.
- **Direct comparison baseline**: `release16_failed_fold_daily_path_diagnostic_v1`, which showed `wf_2020` and `wf_2023_2024` are return-only failures, not drawdown failures.
- **Role**: `diagnose` and target handoff. This notebook creates a non-leaking training target contract for the next checkpoint screen; it does not train, promote, or retain an actor.
- **Target windows**: `wf_2020` and `wf_2023_2024` failed folds, while preserving the passing `wf_2022` guard fold target rows.
- **Changed layer**: `model_selection` / `training_target_composition` / `bc_checkpoint_target_contract`.
- **External reference**: SOXLBacktest qlib buy/sell baseline remains the reference baseline, but no test-window labels are copied into this target.
- **Hypothesis**: the release16 target under-trains participation in the failed folds. A train-window-only overlay can make high-participation states learnable without weakening the `wf_2022` guard target.
- **Acceptance criteria**: all required folds have non-leaking target rows, `wf_2022` target rows are unchanged, failed folds receive overlay rows, and distribution sidecars show the before/after participation lift.
- **Closest historical success**: `rebound_state_factor_target_repair_screen_v1` produced a three-fold non-leaking target contract and BC readiness gate.
- **Closest historical failure**: `release16_cb99_rebound_checkpoint_screen_v1` still failed `wf_2020` and `wf_2023_2024` after checkpoint screening because target participation was too sparse.
- **Why this is not a duplicate**: this notebook revises only the training target contract for failed-fold participation and records overlay rows/audits; it does not rerun the same checkpoint screen or loosen promotion gates.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'SPEC.md').exists():
            return candidate
    raise RuntimeError(f'repo root not found from {start}')

ROOT = find_repo_root(Path.cwd().resolve())
EXPERIMENT = 'release16_failedfold_participation_target_overlay_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_TARGET_CARD_NAME = 'rebound_state_factor_target_repair_screen_v1'
SOURCE_DIAGNOSTIC_CARD_NAME = 'release16_failed_fold_daily_path_diagnostic_v1'
SOURCE_TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_TARGET_CARD_NAME
SOURCE_DIAGNOSTIC_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_DIAGNOSTIC_CARD_NAME
SOURCE_TARGET_PATH = SOURCE_TARGET_CARD / 'combined_target_contract.csv'
SOURCE_TARGET_SUMMARY_PATH = SOURCE_TARGET_CARD / 'summary.json'
SOURCE_DIAGNOSTIC_SUMMARY_PATH = SOURCE_DIAGNOSTIC_CARD / 'summary.json'
SOURCE_DIAGNOSTIC_CONCLUSION_PATH = SOURCE_DIAGNOSTIC_CARD / 'conclusion.md'

REFERENCE_BASELINE = 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02'
DIRECT_COMPARISON_BASELINE = SOURCE_DIAGNOSTIC_CARD_NAME
SOURCE_CHECKPOINT_SCREEN = 'release16_cb99_rebound_checkpoint_screen_v1'
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}
FAILED_FOLDS = ('wf_2020', 'wf_2023_2024')
GUARD_FOLDS = ('wf_2022',)

OVERLAY_RECIPE = {
    'recipe_id': 'release16_failedfold_participation_floor_v1',
    'wf_2020': {
        'position_floor': 0.65,
        'source_high_threshold': 0.50,
        'strategy_drawdown_prev_threshold': 0.05,
        'mom5_prev_threshold': 0.02,
        'description': 'Lift existing high-participation, release/rebound, or drawdown-recovery training states to a learnable 65pct target while excluding downtrend states.',
    },
    'wf_2023_2024': {
        'feature_floor': 0.50,
        'release_floor': 0.65,
        'mom5_prev_threshold': 0.02,
        'mom21_prev_threshold': 0.00,
        'mom21_feature_threshold': 0.12,
        'soxx_mom21_threshold': 0.04,
        'rv21_max': 1.50,
        'description': 'Lift non-downtrend, positive-momentum, low-vol training states to 50pct; lift release/rebound rows to 65pct.',
    },
    'wf_2022': {
        'description': 'Preserve guard-fold target rows unchanged because the source checkpoint screen already passed wf_2022.',
    },
}

SIDE_CARS = {
    'target_contract': CARD / 'combined_target_contract.csv',
    'overlay_rows': CARD / 'target_overlay_rows.csv',
    'distribution': CARD / 'target_distribution_before_after.csv',
    'bucket_delta': CARD / 'target_bucket_delta.csv',
    'scope_audit': CARD / 'target_scope_audit.csv',
    'feature_contrast': CARD / 'target_overlay_feature_contrast.csv',
    'selection': CARD / 'selection.json',
    'summary': CARD / 'summary.json',
    'conclusion': CARD / 'conclusion.md',
}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True) + '\n', encoding='utf-8')

print('card', rel(CARD))
print('source_target', rel(SOURCE_TARGET_PATH))


card runtime/model_selection/card/release16_failedfold_participation_target_overlay_v1
source_target runtime/model_selection/card/rebound_state_factor_target_repair_screen_v1/combined_target_contract.csv


In [2]:
source = pd.read_csv(SOURCE_TARGET_PATH, parse_dates=['datetime', 'train_start', 'train_end', 'eval_start', 'eval_end'])
source_target_summary = load_json(SOURCE_TARGET_SUMMARY_PATH)
source_diagnostic_summary = load_json(SOURCE_DIAGNOSTIC_SUMMARY_PATH)
source_diagnostic_conclusion = SOURCE_DIAGNOSTIC_CONCLUSION_PATH.read_text(encoding='utf-8')

required_columns = {
    'datetime', 'fold', 'risk_target_position', 'rebound_target_position', 'objective_weight',
    'is_downtrend_state', 'release_gate_state', 'drawdown_control_state', 'strategy_drawdown', 'price',
    'train_start', 'train_end', 'eval_start', 'eval_end', 'source_components', 'source_component_count',
}
missing = required_columns - set(source.columns)
assert not missing, missing

print('source rows', len(source), 'folds', sorted(source['fold'].astype(str).unique()))
print('source decision', source_target_summary.get('decision'))
print('diagnostic decision', source_diagnostic_summary.get('decision'))
print('diagnostic next action', source_diagnostic_summary.get('next_mechanism') or source_diagnostic_summary.get('next_action'))


source rows 3731 folds ['wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train']
source decision rebound_state_factor_target_repair_screen_ready_for_checkpoint_no_promotion
diagnostic decision release16_failed_folds_require_participation_target_revision_no_promotion
diagnostic next action build_release16_failedfold_participation_target_overlay_then_rerun_checkpoint_screen


In [3]:
TRUE_VALUES = {'1', 'true', 't', 'yes', 'y'}


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False).astype(bool)
    return series.map(lambda value: str(value).strip().lower() in TRUE_VALUES if pd.notna(value) else False).astype(bool)


def numeric_series(frame: pd.DataFrame, column: str, default: float = np.nan) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype='float64')
    return pd.to_numeric(frame[column], errors='coerce')


def add_train_safe_features(frame: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for fold, group in frame.sort_values(['fold', 'datetime']).groupby('fold', sort=False):
        g = group.copy()
        price = pd.to_numeric(g['price'], errors='coerce')
        strategy_drawdown = pd.to_numeric(g['strategy_drawdown'], errors='coerce').fillna(0.0)
        g['strategy_drawdown_prev_overlay'] = strategy_drawdown.shift(1).fillna(0.0)
        g['soxl_mom5_prev_overlay'] = price.pct_change(5).shift(1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        g['soxl_mom21_prev_overlay'] = price.pct_change(21).shift(1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        prior_price = price.shift(1)
        g['soxl_dd63_prev_overlay'] = (prior_price / prior_price.rolling(63, min_periods=1).max() - 1.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        g['factor_mom21_overlay'] = numeric_series(g, 'mom_21').fillna(g['soxl_mom21_prev_overlay'])
        g['factor_soxx_mom21_overlay'] = numeric_series(g, 'soxx_mom_21').fillna(0.0)
        g['factor_rv21_overlay'] = numeric_series(g, 'rv_21').fillna(np.inf)
        g['factor_ratio_widen_overlay'] = numeric_series(g, 'soxl_soxx_ratio_widen_21').fillna(0.0)
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


def target_distribution(frame: pd.DataFrame, *, target_col: str, label: str) -> pd.DataFrame:
    rows = []
    for fold, group in frame.groupby('fold', sort=True):
        target = pd.to_numeric(group[target_col], errors='coerce')
        rows.append({
            'version': label,
            'fold': str(fold),
            'rows': int(len(group)),
            'start': group['datetime'].min().date().isoformat(),
            'end': group['datetime'].max().date().isoformat(),
            'mean_target_position': float(target.mean()),
            'median_target_position': float(target.median()),
            'max_target_position': float(target.max()),
            'share_ge_30pct': float(target.ge(0.30).mean()),
            'share_ge_50pct': float(target.ge(0.50).mean()),
            'share_ge_65pct': float(target.ge(0.65).mean()),
            'share_ge_70pct': float(target.ge(0.70).mean()),
            'rows_ge_50pct': int(target.ge(0.50).sum()),
            'rows_ge_65pct': int(target.ge(0.65).sum()),
            'rows_ge_70pct': int(target.ge(0.70).sum()),
            'release_rows': int(normalize_bool_series(group.get('release_gate_state', pd.Series(False, index=group.index))).sum()),
            'rebound_rows': int(normalize_bool_series(group.get('is_rebound_cluster', pd.Series(False, index=group.index))).sum()),
            'factor_rebound_rows': int(normalize_bool_series(group.get('factor_rebound_state', pd.Series(False, index=group.index))).sum()) if 'factor_rebound_state' in group.columns else 0,
            'downtrend_rows': int(normalize_bool_series(group.get('is_downtrend_state', pd.Series(False, index=group.index))).sum()),
            'drawdown_control_rows': int(normalize_bool_series(group.get('drawdown_control_state', pd.Series(False, index=group.index))).sum()),
        })
    return pd.DataFrame(rows)


def bucket_label(value: float) -> str:
    if pd.isna(value):
        return 'missing'
    if value < 0.10:
        return '00_cash_lt10'
    if value < 0.30:
        return '10_low_10_30'
    if value < 0.50:
        return '30_mid_30_50'
    if value < 0.65:
        return '50_high_50_65'
    if value < 0.70:
        return '65_high_65_70'
    return '70_full_ge70'


def scope_audit_rows(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        fold = spec['target_fold']
        fold_all = frame[frame['fold'].astype(str).eq(fold)].copy()
        train_start, train_end = pd.Timestamp(spec['train'][0]), pd.Timestamp(spec['train'][1])
        valid_start, valid_end = pd.Timestamp(spec['valid'][0]), pd.Timestamp(spec['valid'][1])
        test_start, test_end = pd.Timestamp(spec['test'][0]), pd.Timestamp(spec['test'][1])
        train_rows = fold_all[fold_all['datetime'].between(train_start, train_end)]
        valid_rows = fold_all[fold_all['datetime'].between(valid_start, valid_end)]
        test_rows = fold_all[fold_all['datetime'].between(test_start, test_end)]
        changed = fold_all[fold_all['target_delta_vs_source'].abs().gt(1e-12)] if 'target_delta_vs_source' in fold_all.columns else pd.DataFrame()
        rows.append({
            'fold': fold_name,
            'target_fold': fold,
            'all_target_rows': int(len(fold_all)),
            'train_target_rows': int(len(train_rows)),
            'valid_overlap_rows': int(len(valid_rows)),
            'test_overlap_rows': int(len(test_rows)),
            'changed_rows_all': int(len(changed)),
            'changed_rows_train': int(len(changed[changed['datetime'].between(train_start, train_end)])) if not changed.empty else 0,
            'train_start': spec['train'][0],
            'train_end': spec['train'][1],
            'valid_start': spec['valid'][0],
            'valid_end': spec['valid'][1],
            'test_start': spec['test'][0],
            'test_end': spec['test'][1],
            'target_contract_gate': bool(len(train_rows) > 0 and len(test_rows) == 0),
        })
    return pd.DataFrame(rows)


In [4]:
for col in ['is_downtrend_state', 'release_gate_state', 'drawdown_control_state', 'factor_rebound_state', 'is_rebound_cluster']:
    if col in source.columns:
        source[col] = normalize_bool_series(source[col])
    else:
        source[col] = False

working = add_train_safe_features(source)
working['source_risk_target_position'] = pd.to_numeric(working['risk_target_position'], errors='coerce').fillna(0.0)
working['source_rebound_target_position'] = pd.to_numeric(working['rebound_target_position'], errors='coerce').fillna(working['source_risk_target_position'])
working['failedfold_participation_overlay_state'] = False
working['failedfold_participation_overlay_reason'] = 'source_contract'
working['failedfold_participation_overlay_floor'] = np.nan
working['failedfold_participation_overlay_recipe_id'] = OVERLAY_RECIPE['recipe_id']
working['target_delta_vs_source'] = 0.0
working['pre_overlay_objective_weight'] = pd.to_numeric(working['objective_weight'], errors='coerce').fillna(1.0)

parts = []
for fold, group in working.groupby('fold', sort=False):
    g = group.copy()
    base = g['source_risk_target_position'].astype(float).clip(0.0, 1.0)
    target = base.copy()
    non_downtrend = ~g['is_downtrend_state'].fillna(False).astype(bool)
    release_or_rebound = (
        g['release_gate_state'].fillna(False).astype(bool)
        | g['factor_rebound_state'].fillna(False).astype(bool)
        | g['is_rebound_cluster'].fillna(False).astype(bool)
    )
    reason = pd.Series('source_contract', index=g.index, dtype='object')
    floor_used = pd.Series(np.nan, index=g.index, dtype='float64')

    if fold == WALK_FORWARD_WINDOWS['wf_2020']['target_fold']:
        recipe = OVERLAY_RECIPE['wf_2020']
        source_high = base.ge(float(recipe['source_high_threshold']))
        recovery = (
            g['strategy_drawdown_prev_overlay'].astype(float).ge(float(recipe['strategy_drawdown_prev_threshold']))
            & g['soxl_mom5_prev_overlay'].astype(float).ge(float(recipe['mom5_prev_threshold']))
        )
        trigger = non_downtrend & (source_high | release_or_rebound | recovery)
        floor = float(recipe['position_floor'])
        target.loc[trigger] = np.maximum(target.loc[trigger], floor)
        reason.loc[trigger & source_high] = 'wf2020_source_high_floor'
        reason.loc[trigger & release_or_rebound] = 'wf2020_release_rebound_floor'
        reason.loc[trigger & recovery] = 'wf2020_recovery_participation_floor'
        floor_used.loc[trigger] = floor
    elif fold == WALK_FORWARD_WINDOWS['wf_2023_2024']['target_fold']:
        recipe = OVERLAY_RECIPE['wf_2023_2024']
        positive_momentum = (
            g['soxl_mom5_prev_overlay'].astype(float).ge(float(recipe['mom5_prev_threshold']))
            & g['soxl_mom21_prev_overlay'].astype(float).ge(float(recipe['mom21_prev_threshold']))
        )
        feature_momentum = (
            g['factor_mom21_overlay'].astype(float).ge(float(recipe['mom21_feature_threshold']))
            | g['factor_soxx_mom21_overlay'].astype(float).ge(float(recipe['soxx_mom21_threshold']))
        )
        low_vol = g['factor_rv21_overlay'].astype(float).le(float(recipe['rv21_max']))
        feature_signal = non_downtrend & low_vol & (positive_momentum | feature_momentum)
        release_signal = non_downtrend & release_or_rebound
        feature_floor = float(recipe['feature_floor'])
        release_floor = float(recipe['release_floor'])
        target.loc[feature_signal] = np.maximum(target.loc[feature_signal], feature_floor)
        target.loc[release_signal] = np.maximum(target.loc[release_signal], release_floor)
        reason.loc[feature_signal] = 'wf2023_positive_momentum_floor'
        reason.loc[release_signal] = 'wf2023_release_rebound_floor'
        floor_used.loc[feature_signal] = feature_floor
        floor_used.loc[release_signal] = release_floor
    else:
        # Keep the wf_2022 guard-fold target unchanged.
        pass

    target = target.clip(0.0, 1.0)
    changed = target.sub(base).abs().gt(1e-12)
    g['failedfold_participation_overlay_state'] = changed
    g['failedfold_participation_overlay_reason'] = reason.where(changed, 'source_contract')
    g['failedfold_participation_overlay_floor'] = floor_used.where(changed, np.nan)
    g['failedfold_participation_target_position'] = target
    g['target_delta_vs_source'] = target - base
    g['risk_target_position'] = target
    g['rebound_target_position'] = np.maximum(g['source_rebound_target_position'].astype(float), target).clip(0.0, 1.0)
    weight = g['pre_overlay_objective_weight'].astype(float).copy()
    weight.loc[changed] = np.maximum(weight.loc[changed], 6.0)
    weight.loc[changed & release_or_rebound] = np.maximum(weight.loc[changed & release_or_rebound], 8.0)
    g['objective_weight'] = weight
    original_components = g['source_components'].fillna('').astype(str)
    g['source_components'] = np.where(
        changed,
        original_components + ',release16_failedfold_participation_overlay_v1',
        original_components,
    )
    original_count = pd.to_numeric(g['source_component_count'], errors='coerce').fillna(0).astype(int)
    g['source_component_count'] = np.where(changed, original_count + 1, original_count)
    g['target_semantics_revision'] = 'release16_failedfold_participation_overlay_v1'
    g['participation_overlay_source_card'] = SOURCE_TARGET_CARD_NAME
    parts.append(g)

overlay = pd.concat(parts, ignore_index=True).sort_values(['fold', 'datetime']).reset_index(drop=True)

base_dist = target_distribution(source.assign(risk_target_position=pd.to_numeric(source['risk_target_position'], errors='coerce')), target_col='risk_target_position', label='source')
overlay_dist = target_distribution(overlay, target_col='risk_target_position', label='overlay')
distribution = pd.concat([base_dist, overlay_dist], ignore_index=True)

overlay_rows = overlay[overlay['failedfold_participation_overlay_state'].astype(bool)].copy()
print('changed rows by fold')
print(overlay_rows.groupby('fold').size().to_string())
print('\nbefore/after distribution')
display(distribution)


changed rows by fold
fold
wf_2020_train         699
wf_2023_2024_train    193

before/after distribution


,version,fold,rows,start,end,mean_target_position,median_target_position,max_target_position,share_ge_30pct,share_ge_50pct,share_ge_65pct,share_ge_70pct,rows_ge_50pct,rows_ge_65pct,rows_ge_70pct,release_rows,rebound_rows,factor_rebound_rows,downtrend_rows,drawdown_control_rows
0,source,wf_2020_train,2470,2010-03-11,2019-12-31,0.312066,0.250000,0.9155,0.383806,0.246154,0.015789,0.008502,608,39,21,42,42,0,781,781
1,source,wf_2022_train,505,2020-01-02,2021-12-31,0.198093,0.146277,1.0000,0.146535,0.067327,0.043564,0.043564,34,22,22,9,13,9,234,0
2,source,wf_2023_2024_train,756,2020-01-02,2022-12-30,0.154589,0.105063,1.0000,0.100529,0.044974,0.029101,0.029101,34,22,22,11,14,11,478,0
3,overlay,wf_2020_train,2470,2010-03-11,2019-12-31,0.357942,0.250000,0.9155,0.408097,0.303239,0.298785,0.008502,749,738,21,42,42,0,781,781
4,overlay,wf_2022_train,505,2020-01-02,2021-12-31,0.198093,0.146277,1.0000,0.146535,0.067327,0.043564,0.043564,34,22,22,9,13,9,234,0
5,overlay,wf_2023_2024_train,756,2020-01-02,2022-12-30,0.236502,0.122797,1.0000,0.322751,0.300265,0.034392,0.029101,227,26,22,11,14,11,478,0


In [5]:
# Build sidecar audits.
source_bucket = source[['datetime', 'fold', 'risk_target_position']].copy()
source_bucket['version'] = 'source'
source_bucket['bucket'] = source_bucket['risk_target_position'].astype(float).map(bucket_label)
overlay_bucket = overlay[['datetime', 'fold', 'risk_target_position']].copy()
overlay_bucket['version'] = 'overlay'
overlay_bucket['bucket'] = overlay_bucket['risk_target_position'].astype(float).map(bucket_label)
bucket_counts = pd.concat([source_bucket, overlay_bucket], ignore_index=True).groupby(['fold', 'version', 'bucket']).size().rename('rows').reset_index()
bucket_pivot = bucket_counts.pivot_table(index=['fold', 'bucket'], columns='version', values='rows', fill_value=0).reset_index()
for column in ['source', 'overlay']:
    if column not in bucket_pivot.columns:
        bucket_pivot[column] = 0
bucket_pivot['delta_rows'] = bucket_pivot['overlay'].astype(int) - bucket_pivot['source'].astype(int)

feature_cols = [
    'source_risk_target_position',
    'risk_target_position',
    'target_delta_vs_source',
    'strategy_drawdown_prev_overlay',
    'soxl_mom5_prev_overlay',
    'soxl_mom21_prev_overlay',
    'soxl_dd63_prev_overlay',
    'factor_mom21_overlay',
    'factor_soxx_mom21_overlay',
    'factor_rv21_overlay',
    'factor_ratio_widen_overlay',
    'mom_5',
    'mom_21',
    'drawdown_63',
    'rv_21',
    'soxx_mom_21',
    'soxl_soxx_ratio_widen_21',
    'soxl_soxx_ratio_z_252',
]
contrast_rows = []
for fold, group in overlay.groupby('fold', sort=True):
    changed = group['failedfold_participation_overlay_state'].astype(bool)
    for feature in feature_cols:
        if feature not in group.columns:
            continue
        values = pd.to_numeric(group[feature], errors='coerce')
        changed_values = values[changed]
        unchanged_values = values[~changed]
        contrast_rows.append({
            'fold': fold,
            'feature': feature,
            'overlay_changed_rows': int(changed.sum()),
            'overlay_unchanged_rows': int((~changed).sum()),
            'overlay_changed_mean': float(changed_values.mean()) if changed_values.notna().any() else np.nan,
            'overlay_unchanged_mean': float(unchanged_values.mean()) if unchanged_values.notna().any() else np.nan,
            'changed_minus_unchanged': (float(changed_values.mean()) - float(unchanged_values.mean())) if changed_values.notna().any() and unchanged_values.notna().any() else np.nan,
        })
feature_contrast = pd.DataFrame(contrast_rows)

scope_audit = scope_audit_rows(overlay)
target_contract_gate = bool(scope_audit['target_contract_gate'].all())
failed_fold_overlay_counts = {
    fold_name: int(overlay_rows[overlay_rows['fold'].astype(str).eq(WALK_FORWARD_WINDOWS[fold_name]['target_fold'])].shape[0])
    for fold_name in FAILED_FOLDS
}
guard_fold_changed_rows = {
    fold_name: int(overlay_rows[overlay_rows['fold'].astype(str).eq(WALK_FORWARD_WINDOWS[fold_name]['target_fold'])].shape[0])
    for fold_name in GUARD_FOLDS
}
readiness_checks = {
    'target_contract_gate': target_contract_gate,
    'wf_2020_overlay_rows_gt0': failed_fold_overlay_counts['wf_2020'] > 0,
    'wf_2023_2024_overlay_rows_gt0': failed_fold_overlay_counts['wf_2023_2024'] > 0,
    'wf_2022_guard_unchanged': guard_fold_changed_rows['wf_2022'] == 0,
}
readiness_gate = bool(all(readiness_checks.values()))

assert target_contract_gate, scope_audit
assert failed_fold_overlay_counts['wf_2020'] > 0, failed_fold_overlay_counts
assert failed_fold_overlay_counts['wf_2023_2024'] > 0, failed_fold_overlay_counts
assert guard_fold_changed_rows['wf_2022'] == 0, guard_fold_changed_rows

# Keep the old source positions explicitly for audit before writing the revised contract.
ordered_columns = list(source.columns)
new_columns = [column for column in overlay.columns if column not in ordered_columns]
overlay_to_write = overlay[ordered_columns + new_columns].copy()
overlay_to_write.to_csv(SIDE_CARS['target_contract'], index=False)
overlay_rows.to_csv(SIDE_CARS['overlay_rows'], index=False)
distribution.to_csv(SIDE_CARS['distribution'], index=False)
bucket_pivot.to_csv(SIDE_CARS['bucket_delta'], index=False)
scope_audit.to_csv(SIDE_CARS['scope_audit'], index=False)
feature_contrast.to_csv(SIDE_CARS['feature_contrast'], index=False)

print('readiness_checks', readiness_checks)
print('target_contract', rel(SIDE_CARS['target_contract']))
display(scope_audit)
display(bucket_pivot)


readiness_checks {'target_contract_gate': True, 'wf_2020_overlay_rows_gt0': True, 'wf_2023_2024_overlay_rows_gt0': True, 'wf_2022_guard_unchanged': True}
target_contract runtime/model_selection/card/release16_failedfold_participation_target_overlay_v1/combined_target_contract.csv


,fold,target_fold,all_target_rows,train_target_rows,valid_overlap_rows,test_overlap_rows,changed_rows_all,changed_rows_train,train_start,train_end,valid_start,valid_end,test_start,test_end,target_contract_gate
0,wf_2020,wf_2020_train,2470,2218,252,0,699,619,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,True
1,wf_2022,wf_2022_train,505,253,252,0,0,0,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,True
2,wf_2023_2024,wf_2023_2024_train,756,756,0,0,193,193,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,True


version,fold,bucket,overlay,source,delta_rows
0,wf_2020_train,00_cash_lt10,287.0,309.0,-22
1,wf_2020_train,10_low_10_30,1175.0,1213.0,-38
2,wf_2020_train,30_mid_30_50,259.0,340.0,-81
3,wf_2020_train,50_high_50_65,11.0,569.0,-558
4,wf_2020_train,65_high_65_70,717.0,18.0,699
5,wf_2020_train,70_full_ge70,21.0,21.0,0
6,wf_2022_train,00_cash_lt10,168.0,168.0,0
7,wf_2022_train,10_low_10_30,263.0,263.0,0
8,wf_2022_train,30_mid_30_50,40.0,40.0,0
9,wf_2022_train,50_high_50_65,12.0,12.0,0


In [6]:
summary_distribution = distribution.pivot_table(
    index='fold',
    columns='version',
    values=['mean_target_position', 'share_ge_50pct', 'share_ge_65pct', 'share_ge_70pct', 'rows_ge_50pct', 'rows_ge_65pct'],
    aggfunc='first',
)
summary_distribution.columns = [f'{metric}_{version}' for metric, version in summary_distribution.columns]
summary_distribution = summary_distribution.reset_index()

summary_distribution['mean_target_position_delta'] = summary_distribution['mean_target_position_overlay'] - summary_distribution['mean_target_position_source']
summary_distribution['share_ge_50pct_delta'] = summary_distribution['share_ge_50pct_overlay'] - summary_distribution['share_ge_50pct_source']
summary_distribution['share_ge_65pct_delta'] = summary_distribution['share_ge_65pct_overlay'] - summary_distribution['share_ge_65pct_source']

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'release16_failedfold_participation_target_overlay_ready_for_checkpoint_no_promotion' if readiness_gate else 'release16_failedfold_participation_target_overlay_blocked_no_promotion',
    'canonical_baseline': 'retained_soxl_harness_model_frontier',
    'direct_comparison_baseline': DIRECT_COMPARISON_BASELINE,
    'source_card': SOURCE_TARGET_CARD_NAME,
    'source_failed_fold_experiment': SOURCE_DIAGNOSTIC_CARD_NAME,
    'source_checkpoint_screen': SOURCE_CHECKPOINT_SCREEN,
    'source_target_contract': rel(SOURCE_TARGET_PATH),
    'source_target_decision': source_target_summary.get('decision'),
    'source_diagnostic_decision': source_diagnostic_summary.get('decision'),
    'reference_baseline': REFERENCE_BASELINE,
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_target_composition', 'bc_checkpoint_target_contract'],
    'target_window': list(FAILED_FOLDS),
    'accepted_tradeoff': 'Increase failed-fold training participation targets while preserving the wf_2022 guard-fold target contract unchanged.',
    'hypothesis': 'Failed folds are return-only underparticipation failures, so target rows need higher train-window participation labels before another checkpoint screen.',
    'acceptance_criteria': [
        'all required folds have target rows and no test overlap',
        'wf_2022 guard-fold target rows are unchanged',
        'wf_2020 and wf_2023_2024 receive overlay target rows',
        'no actor promotion or retention occurs from this card',
    ],
    'failure_mode': 'If the next checkpoint replay still fails return gates, the target overlay is insufficient and model/reward architecture needs revision rather than further target relabeling.',
    'closest_historical_success_case': 'rebound_state_factor_target_repair_screen_v1 stitched non-leaking three-fold targets and passed BC readiness gates.',
    'closest_historical_failure_case': 'release16_cb99_rebound_checkpoint_screen_v1 failed wf_2020 and wf_2023_2024 after checkpoint screening because participation targets remained sparse.',
    'why_not_duplicate': 'This card writes a revised failed-fold participation target contract and audits overlay rows; it does not rerun the same checkpoint screen.',
    'overlay_recipe': OVERLAY_RECIPE,
    'target_contract_gate': target_contract_gate,
    'target_revision_ready': readiness_gate,
    'next_training_allowed_after_target_revision': readiness_gate,
    'checkpoint_replay_allowed_next': readiness_gate,
    'bc_readiness_gate': False,
    'bc_screen_allowed_next': False,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'readiness_checks': readiness_checks,
    'failed_fold_overlay_counts': failed_fold_overlay_counts,
    'guard_fold_changed_rows': guard_fold_changed_rows,
    'distribution_summary': summary_distribution.to_dict('records'),
    'target_contract_path': rel(SIDE_CARS['target_contract']),
    'target_overlay_rows_path': rel(SIDE_CARS['overlay_rows']),
    'target_distribution_path': rel(SIDE_CARS['distribution']),
    'target_bucket_delta_path': rel(SIDE_CARS['bucket_delta']),
    'target_scope_audit_path': rel(SIDE_CARS['scope_audit']),
    'target_feature_contrast_path': rel(SIDE_CARS['feature_contrast']),
    'next_mechanism': 'rerun_release16_checkpoint_screen_with_failedfold_participation_overlay' if readiness_gate else 'fix_target_overlay_scope_or_guard_fold_before_checkpoint_screen',
    'primary_blocker': 'target_revision_ready' if readiness_gate else 'target_overlay_blocked',
}
summary = dict(selection)
summary['sidecars'] = {name: rel(path) for name, path in SIDE_CARS.items()}
summary['summary'] = {
    'source_rows': int(len(source)),
    'overlay_rows': int(len(overlay_rows)),
    'target_contract_gate': target_contract_gate,
    'readiness_gate': readiness_gate,
    'failed_fold_overlay_counts': failed_fold_overlay_counts,
    'guard_fold_changed_rows': guard_fold_changed_rows,
}

write_json(SIDE_CARS['selection'], selection)
write_json(SIDE_CARS['summary'], summary)
print(json.dumps(selection, indent=2, sort_keys=True)[:5000])
display(summary_distribution)


{
  "acceptance_criteria": [
    "all required folds have target rows and no test overlap",
    "wf_2022 guard-fold target rows are unchanged",
    "wf_2020 and wf_2023_2024 receive overlay target rows",
    "no actor promotion or retention occurs from this card"
  ],
  "accepted_tradeoff": "Increase failed-fold training participation targets while preserving the wf_2022 guard-fold target contract unchanged.",
  "actor_retention_allowed": false,
  "bc_readiness_gate": false,
  "bc_screen_allowed_next": false,
  "canonical_baseline": "retained_soxl_harness_model_frontier",
  "changed_layer": [
    "model_selection",
    "training_target_composition",
    "bc_checkpoint_target_contract"
  ],
  "checkpoint_replay_allowed_next": true,
  "closest_historical_failure_case": "release16_cb99_rebound_checkpoint_screen_v1 failed wf_2020 and wf_2023_2024 after checkpoint screening because participation targets remained sparse.",
  "closest_historical_success_case": "rebound_state_factor_target_rep

,fold,mean_target_position_overlay,mean_target_position_source,rows_ge_50pct_overlay,rows_ge_50pct_source,rows_ge_65pct_overlay,rows_ge_65pct_source,share_ge_50pct_overlay,share_ge_50pct_source,share_ge_65pct_overlay,share_ge_65pct_source,share_ge_70pct_overlay,share_ge_70pct_source,mean_target_position_delta,share_ge_50pct_delta,share_ge_65pct_delta
0,wf_2020_train,0.357942,0.312066,749,608,738,39,0.303239,0.246154,0.298785,0.015789,0.008502,0.008502,0.045877,0.057085,0.282996
1,wf_2022_train,0.198093,0.198093,34,34,22,22,0.067327,0.067327,0.043564,0.043564,0.043564,0.043564,0.000000,0.000000,0.000000
2,wf_2023_2024_train,0.236502,0.154589,227,34,26,22,0.300265,0.044974,0.034392,0.029101,0.029101,0.029101,0.081913,0.255291,0.005291


In [7]:
selection = load_json(SIDE_CARS['selection'])
distribution = pd.read_csv(SIDE_CARS['distribution'])
summary_distribution = distribution.pivot_table(
    index='fold',
    columns='version',
    values=['mean_target_position', 'share_ge_50pct', 'share_ge_65pct', 'share_ge_70pct'],
    aggfunc='first',
)
summary_distribution.columns = [f'{metric}_{version}' for metric, version in summary_distribution.columns]
summary_distribution = summary_distribution.reset_index()
lines = [
    '# Release16 Failed-Fold Participation Target Overlay',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    f"Source target: `{selection['source_target_contract']}` from `{selection['source_card']}`.",
    f"Direct comparison: `{selection['direct_comparison_baseline']}`; reference baseline `{selection['reference_baseline']}`.",
    '',
    'Overlay result:',
]
for row in summary_distribution.sort_values('fold').itertuples(index=False):
    fold = row.fold
    source_mean = getattr(row, 'mean_target_position_source')
    overlay_mean = getattr(row, 'mean_target_position_overlay')
    source_ge50 = getattr(row, 'share_ge_50pct_source')
    overlay_ge50 = getattr(row, 'share_ge_50pct_overlay')
    source_ge65 = getattr(row, 'share_ge_65pct_source')
    overlay_ge65 = getattr(row, 'share_ge_65pct_overlay')
    lines.append(
        f"- {fold}: mean {source_mean:.3f} -> {overlay_mean:.3f}; share >=50% {source_ge50:.3f} -> {overlay_ge50:.3f}; share >=65% {source_ge65:.3f} -> {overlay_ge65:.3f}."
    )
lines.extend([
    '',
    f"Readiness checks: `{selection['readiness_checks']}`.",
    f"Failed-fold overlay counts: `{selection['failed_fold_overlay_counts']}`; guard-fold changed rows: `{selection['guard_fold_changed_rows']}`.",
    '',
    'Risk note: this card revises only train-window target labels. It does not use test-window labels, train an actor, promote a model, or relax the required walk-forward return/drawdown gates.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `combined_target_contract.csv`, `target_overlay_rows.csv`, `target_distribution_before_after.csv`, `target_bucket_delta.csv`, `target_scope_audit.csv`, `target_overlay_feature_contrast.csv`, `selection.json`, `summary.json`.',
])
conclusion = '\n'.join(lines).rstrip() + '\n'
SIDE_CARS['conclusion'].write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))


# Release16 Failed-Fold Participation Target Overlay

Decision: `release16_failedfold_participation_target_overlay_ready_for_checkpoint_no_promotion`.

Source target: `runtime/model_selection/card/rebound_state_factor_target_repair_screen_v1/combined_target_contract.csv` from `rebound_state_factor_target_repair_screen_v1`.
Direct comparison: `release16_failed_fold_daily_path_diagnostic_v1`; reference baseline `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02`.

Overlay result:
- wf_2020_train: mean 0.312 -> 0.358; share >=50% 0.246 -> 0.303; share >=65% 0.016 -> 0.299.
- wf_2022_train: mean 0.198 -> 0.198; share >=50% 0.067 -> 0.067; share >=65% 0.044 -> 0.044.
- wf_2023_2024_train: mean 0.155 -> 0.237; share >=50% 0.045 -> 0.300; share >=65% 0.029 -> 0.034.

Readiness checks: `{'target_contract_gate': True, 'wf_2020_overlay_rows_gt0': True, 'wf_2022_guard_unchanged': True, 'wf_2023_2024_overlay_rows_gt0': True}`.
Failed-fold overlay counts: `{'wf_2020': 699, 'wf_2023_2024': 193}`; guard-fold changed rows: `{'wf_2022': 0}`.

Risk note: this card revises only train-window target labels. It does not use test-window labels, train an actor, promote a model, or relax the required walk-forward return/drawdown gates.

Next action: `rerun_release16_checkpoint_screen_with_failedfold_participation_overlay`.

Artifacts: `combined_target_contract.csv`, `target_overlay_rows.csv`, `target_distribution_before_after.csv`, `target_bucket_delta.csv`, `target_scope_audit.csv`, `target_overlay_feature_contrast.csv`, `selection.json`, `summary.json`.
